<a href="https://colab.research.google.com/github/jinalan-g/rd/blob/test/Trailer_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Install required Libraries
!pip install moviepy openai-whisper --quiet
import os
import whisper
from moviepy.editor import VideoFileClip, concatenate_videoclips
import requests
import json
import ast
import ssl
from google.colab import userdata
ssl._create_default_https_context = ssl._create_unverified_context

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 9.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.2 MB/

  if event.key is 'enter':



In [ ]:
# @title Transcribe Video Function
def transcribe_video(video_path, model_name="turbo"):
    model = whisper.load_model(model_name)
    audio_path = "temp_audio.wav"
    os.system(f"ffmpeg -y -i '{video_path}' -vn -acodec pcm_s16le -ar 16000 -ac 2 '{audio_path}'")
    result = model.transcribe(audio_path)
    transcription = []
    for segment in result["segments"]:
      print(segment)
      transcription.append(
            {
                "start": segment["start"],
                "end": segment["end"],
                "text": segment["text"].strip(),
            }
        )
      #print(transcription)
    return transcription

In [ ]:
# @title Use Gemini to get relevent segements
API_KEY = userdata.get('GEMINI_API_KEY')
def get_relevant_segments(transcript, user_query):
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
    prompt = f"""You are an expert video editor who can read video transcripts and perform video editing. Given a transcript with segments, your task is to identify all the conversations related to a user query. Follow these guidelines when choosing conversations. A group of continuous segments in the transcript is a conversation.

Guidelines:
1. The conversation should be relevant to the user query. The conversation should include more than one segment to provide context and continuity.
2. The conversation should not cut off in the middle of a sentence or idea.
3. Choose single conversation from the transcript that is relevant to the user query.
4. Match the start and end time of the conversations using the segment timestamps from the transcript.
5. The conversations should be a direct part of the video and should not be out of context.
6. Last part of the video should be related to Downloading the Pocket FM App.
7. Most important: Exclude the parts where there is any kind of sensitive contents, that would violet google ads video policy.
Segment should be excluded if it belongs to below areas:
**Area 1: Inappropriate Content**
**Dangerous or Derogatory Content:**
    * **Hate Speech:** Does it incite hatred, promote discrimination, or disparage individuals/groups based on race, ethnic origin, religion, disability, age, nationality, veteran status, sexual orientation, gender, gender identity? Does it promote hate groups or paraphernalia?
    * **Harassment & Bullying:** Does it single out an individual for abuse? Does it suggest a tragic event did not happen (e.g., "crisis actor" claims)?
    * **Threats & Self-Harm:** Does it threaten or advocate for physical/mental harm on oneself or others? Does it advocate suicide or anorexia? Does it glorify violence or promote terrorist/criminal organizations?
    * **Exploitation:** Does it feature extortion, blackmail, or soliciting/promoting dowries?
**Shocking Content:**
    * **Violence:** Is the focal point graphic violence, blood, and gore for shock value? Is there a focus on characters in pain/distress? Does it show dismemberment, decapitation, torture, or sexual violence?
    * **Blood and Gore:** Are there gratuitous portrayals of bodily fluids or waste (blood, guts, gore, sexual fluids)? Are there depictions of corpses, dead bodies, or exposed organs/internal tissue?
    * **Horror and Fear:** Is the ONLY focus to frighten users? Does it contain "jumpscares"?
**Sensitive Events:** Does it exploit, dismiss, or condone unforeseen events like civil emergencies, natural disasters, public health emergencies, terrorism, or conflict?
**Animal Cruelty:** Does it promote cruelty for entertainment (e.g., "cock or dog fighting")? Does it trade in products from threatened/extinct species (ivory, tiger skins, shark fins, rhino horn)?
**Adult Themes in Family Content:** Is the content made to appear family-friendly but contains adult themes, violence, vulgarity, or sexual content involving popular children's characters? This includes "fetishy" content associated with gluttony, feet, being tied up, or bodily fluids involving children's characters.
**Enabling Dishonest Behavior:** Does it promote hacking software, services to artificially inflate traffic, fake documents, or academic cheating services?

**Area 2: Sexual Content**
**Strongly Restricted Categories:**
    * **Nudity:** Depictions of exposed intimate body parts (genitals, female breasts, buttocks), including blurred, censored, or artistic representations. This includes fully exposed buttocks with a visible butt crack and post-production coverage like stars or pixelation.
    * **Sexual Encounter Dating:** Dating explicitly motivated by sexual encounters, like hook-up or swinger sites.
**Moderately Restricted Categories:**
    * **Partial Nudity:** Lower or outer curve of the female breast exposed; majority of buttocks exposed.
    * **Sexually Suggestive Elements:** Provocative poses; unnecessary focus on individual body parts (even if clothed); mimicking sexual positions; sensual dancing in private; taking off clothing in a sexual context.
    * **Compensated Sexual Acts:** 'Sugar' dating or any arrangement where compensation is exchanged for sexual acts.
    * **Mail-Order Brides:** Promotion of services to find a spouse from another country via a catalog.

**Area 3: Inappropriate Language (Full Tiers)**
Identify any profane or obscene language. Classify its severity based on the following complete tiers:
    * **Tier 1 (None):** "darn, dang, shoot, bottom, heck, pee, poo, crud, jerk, freaking, booty, butt"
    * **Tier 2 (Mild):** "hell, ass, damn, boobs, crap, pissed, bastard, jackass, prick, arse"
    * **Tier 3 (Moderate):** "shit, tits, bullshit, asshole, titties, douchebag, wanker, creampie"
    * **Tier 4 (Heavy):** "dick, cock, fuck, cum, fanny, pussy, cooch, motherfucker, bitch, whore, slut, twat, cunt, kunt"
    * **Tier 5 (Very Heavy):** Any racial, ethnic, or sexual slurs.
**Censored Profanity Detection:** You MUST analyze the visual stream. If audio profanity is censored (e.g., 'bleeped out'), you must determine if the speaker's mouth movements still clearly form the profane word. If the word is discernible from the mouthing, it is a violation.
**Multilingual & Synonym Detection:** Your analysis is not limited to the exact English words listed. You must identify synonyms and common misspelllings. You must also attempt to identify the primary language of the video and apply the same level of scrutiny for profane terms in that language.

Output format: {{ "conversations": [{{"start": "s1", "end": "e1"}}, {{"start": "s2", "end": "e2"}}] }}

Transcript:
{transcript}

User query:
{user_query}"""

    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-exp:generateContent?key={userdata.get('GEMINI_API_KEY')}"
    headers = {"Content-Type": "application/json"}

    data = {
        "contents": [{"role": "user", "parts": [{"text": prompt}]}],
        "generationConfig": {
            "temperature": 1,
            "topK": 40,
            "topP": 0.95,
            "maxOutputTokens": 8192,
            "responseMimeType": "text/plain",
        },
    }

    response = requests.post(url, headers=headers, json=data)
    data =  response.json()["candidates"][0]["content"]["parts"][0]["text"]
    data = data.replace("```json","").replace("```","")
    conversations = ast.literal_eval(data)["conversations"]
    return conversations

In [ ]:
# @title Edit Video Function

def edit_video(original_video_path, segments, output_video_path, fade_duration=0.2):
    video = VideoFileClip(original_video_path)
    clips = []
    for seg in segments:
        start = seg["start"]
        end = seg["end"]
        clip = video.subclip(start, end).fadein(fade_duration).fadeout(fade_duration)
        clips.append(clip)
    if clips:
        final_clip = concatenate_videoclips(clips, method="compose")
        final_clip.write_videofile(
            output_video_path, codec="libx264", audio_codec="aac"
        )
    else:
        print("No segments to include in the edited video.")


In [ ]:
# @title Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title Enter Input video path
input_video = input("Enter the path to your input video in Google Drive (e.g., /content/drive/MyDrive/my_video.mp4): ")
output_video = "edited_output.mp4"

# Step 1: Transcribe
print("Transcribing video...")
transcription = transcribe_video(input_video, model_name="turbo")

Enter the path to your input video in Google Drive (e.g., /content/drive/MyDrive/my_video.mp4): /content/drive/MyDrive/Dubbing videos/Tira/Beauty for every vibe and every skin type _ Only on Tira _ #foreveryyou.mp4
Transcribing video...


  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



{'id': 0, 'seek': 0, 'start': 0.0, 'end': 13.76, 'text': ' Long weekend. Goa, goa. Goa? That means whole new skincare. 50 vacation reels, 100 influencer tips, unlock a girl, lock a sulphate free, cruelty free, buy one, get one free.', 'tokens': [50365, 8282, 6711, 13, 1037, 64, 11, 352, 64, 13, 1037, 64, 30, 663, 1355, 1379, 777, 29461, 13, 2625, 12830, 319, 1625, 11, 2319, 39503, 6082, 11, 11634, 257, 2013, 11, 4017, 257, 47286, 473, 1737, 11, 40145, 1737, 11, 2256, 472, 11, 483, 472, 1737, 13, 51053], 'temperature': 0.0, 'avg_logprob': -0.34744197621065026, 'compression_ratio': 1.4439024390243902, 'no_speech_prob': 1.975477594007291e-11}
{'id': 1, 'seek': 0, 'start': 15.9, 'end': 28.240000000000002, 'text': " Let's go next month. Bro, Teera's got you. Beauty for every vibe and every skin type. Only on Teera. Download the app now.", 'tokens': [51160, 961, 311, 352, 958, 1618, 13, 5425, 11, 1989, 1663, 311, 658, 291, 13, 21450, 337, 633, 14606, 293, 633, 3178, 2010, 13, 5686, 322, 1989

In [ ]:
# @title Save Transcript to File
import os

# Construct the output filename from the input video path
video_name = os.path.splitext(os.path.basename(input_video))[0]
output_transcript_path = f"/content/drive/MyDrive/Transcripts/{video_name}_transcript.txt"

# Create the directory if it doesn't exist
output_dir = os.path.dirname(output_transcript_path)
os.makedirs(output_dir, exist_ok=True)

# Write the transcription to the file
with open(output_transcript_path, "w") as f:
    for segment in transcription:
        f.write(f"[{segment['start']:.2f}-{segment['end']:.2f}] {segment['text']}\n")

print(f"Transcript saved to: {output_transcript_path}")

Transcript saved to: /content/drive/MyDrive/Transcripts/Beauty for every vibe and every skin type _ Only on Tira _ #foreveryyou_transcript.txt


In [ ]:
# @title Enter prompt to create Trailer

# User Query
user_query = input("Enter your query: ")

relevant_segments = get_relevant_segments(transcription, user_query)
edit_video(input_video, relevant_segments, output_video)

In [ ]:
# @title Enter the path to save output vider

output_video = input("Enter the path to save the edited video: ")
edit_video(input_video, relevant_segments, output_video)